In [ ]:
from google.colab import userdata, drive
import os

token = userdata.get("GITHUB_TOKEN")

if os.path.exists('/content/musicandmemory'):
    %cd /content/musicandmemory
    !git pull https://{token}@github.com/anikanb-32/musicandmemory.git
else:
    !git clone https://{token}@github.com/anikanb-32/musicandmemory.git
    %cd /content/musicandmemory

print("Current directory:", os.getcwd())

/content/musicandmemory
From https://github.com/anikanb-32/musicandmemory
 * branch            HEAD       -> FETCH_HEAD
Already up to date.
Current directory: /content/musicandmemory


In [ ]:
drive.mount('/content/drive')
!pip install -r requirements.txt

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# load APIs

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["SPOTIFY_CLIENT_ID"] = userdata.get("SPOTIFY_CLIENT_ID")
os.environ["SPOTIFY_CLIENT_SECRET"] = userdata.get("SPOTIFY_CLIENT_SECRET")

DATA_PATH = '/content/drive/MyDrive/'

In [ ]:
# load billboard data

import pandas as pd

df_billboard = pd.read_csv(DATA_PATH + "charts.csv")
print(f"Total rows: {len(df_billboard)}")
print(f"Columns: {list(df_billboard.columns)}")
df_billboard.head()

Total rows: 330087
Columns: ['date', 'rank', 'song', 'artist', 'last-week', 'peak-rank', 'weeks-on-board']


,date,rank,song,artist,last-week,peak-rank,weeks-on-board
0,2021-11-06,1,Easy On Me,Adele,1.0,1,3
1,2021-11-06,2,Stay,The Kid LAROI & Justin Bieber,2.0,1,16
2,2021-11-06,3,Industry Baby,Lil Nas X & Jack Harlow,3.0,1,14
3,2021-11-06,4,Fancy Like,Walker Hayes,4.0,3,19
4,2021-11-06,5,Bad Habits,Ed Sheeran,5.0,2,18


In [ ]:
# cleaning data

# standardize column names
df_billboard.columns = [col.strip().lower().replace(" ", "_").replace("-", "_") for col in df_billboard.columns]

# convert date and extract year
df_billboard['date'] = pd.to_datetime(df_billboard['date'])
df_billboard['year'] = df_billboard['date'].dt.year

# deduplicate keeping best chart position per song
df_deduped = (
    df_billboard
    .sort_values('peak_rank')
    .drop_duplicates(subset=['song', 'artist'], keep='first')
    .reset_index(drop=True)
)

print(f"Before dedup: {len(df_billboard)} rows")
print(f"After dedup:  {len(df_deduped)} unique songs")

df_deduped.to_csv(DATA_PATH + "billboard_deduped.csv", index=False)

Before dedup: 330087 rows
After dedup:  29681 unique songs


In [ ]:
# MusicBrainz

import requests
import time
from tqdm import tqdm

# note: Replacing Spotify with MusicBrainz due to:
# Spotify audio_features endpoint returns HTTP 403 for new apps since late 2024
# Spotify search hit rate limits after ~550 requests :(
# MusicBrainz is free, no API key required, better coverage for non-Western and older music

HEADERS = {"User-Agent": "MusicAndMemory/1.0 (your@email.com)"}

def search_musicbrainz(title, artist):
    try:
        url = "https://musicbrainz.org/ws/2/recording/"
        params = {
            "query": f'recording:"{title}" AND artist:"{artist}"',
            "fmt": "json",
            "limit": 1
        }
        response = requests.get(url, params=params, headers=HEADERS)
        data = response.json()

        if data.get("recordings"):
            rec = data["recordings"][0]
            releases = rec.get("releases", [])
            country = releases[0].get("country") if releases else None
            date = releases[0].get("date") if releases else None

            return {
                "mb_id": rec.get("id"),
                "mb_title": rec.get("title"),
                "mb_artist": rec["artist-credit"][0]["name"] if rec.get("artist-credit") else None,
                "mb_country": country,
                "mb_date": date,
                "mb_score": rec.get("score"),
            }
    except Exception as e:
        print(f"Error searching for {title} by {artist}: {e}")
    return None

# Test first
test = search_musicbrainz("Hello", "Adele")
print(test)

{'mb_id': '623b39c5-3cf1-49b1-83a7-a22362f3b9a6', 'mb_title': 'Hello', 'mb_artist': 'Adele', 'mb_country': 'XW', 'mb_date': '2023-10-12', 'mb_score': 100}


In [ ]:
# MusicBrainz will take 9 hours to run the full loop, trying to only pull relevant times

print(f"Year range: {df_deduped['year'].min()} to {df_deduped['year'].max()}")
print(df_deduped['year'].value_counts().sort_index())

Year range: 1958 to 2021
year
1958    280
1959    563
1960    586
1961    687
1962    670
       ... 
2017    463
2018    594
2019    509
2020    676
2021    647
Name: count, Length: 64, dtype: int64


In [ ]:
# MusicBrainz - filtered approach

# dementia patients in care today are roughly 70-90 years old
# born approximately 1935-1955
# reminiscence bump (ages 15-25) = roughly 1950-1980
# adding a buffer decade on each side for safety

YEAR_MIN = 1950
YEAR_MAX = 1990

df_filtered = df_deduped[
    (df_deduped['year'] >= YEAR_MIN) &
    (df_deduped['year'] <= YEAR_MAX)
].reset_index(drop=True)

print(f"Full dataset: {len(df_deduped)} songs")
print(f"Filtered to {YEAR_MIN}-{YEAR_MAX}: {len(df_filtered)} songs")
print(f"\nSongs per decade:")
df_filtered['decade'] = (df_filtered['year'] // 10) * 10
print(df_filtered['decade'].value_counts().sort_index())

Full dataset: 29681 songs
Filtered to 1950-1990: 17470 songs

Songs per decade:
decade
1950     843
1960    6852
1970    5279
1980    4115
1990     381
Name: count, dtype: int64


In [ ]:
# running the filtered MusicBrainz loop

# sample top 500 songs per decade for manageable runtime
df_filtered = (
    df_filtered
    .sort_values('peak_rank')
    .groupby('decade')
    .head(500)
    .reset_index(drop=True)
)
print(f"Sampled dataset: {len(df_filtered)} songs")

# Run MusicBrainz on sampled set
mb_data = []
batch_size = 500

for i, row in tqdm(df_filtered.iterrows(), total=len(df_filtered)):
    result = search_musicbrainz(row["song"], row["artist"])
    if result:
        result["billboard_index"] = i
        mb_data.append(result)

    time.sleep(1.1)

    if (i + 1) % batch_size == 0:
        pd.DataFrame(mb_data).to_csv(
            DATA_PATH + f"mb_partial_{i+1}.csv", index=False
        )
        print(f"Saved progress at row {i+1}")

df_mb = pd.DataFrame(mb_data)
print(f"Matched {len(df_mb)} / {len(df_filtered)} songs")
df_mb.to_csv(DATA_PATH + "mb_enriched.csv", index=False)

Sampled dataset: 2381 songs


 21%|██        | 500/2381 [13:16<52:04,  1.66s/it]

Saved progress at row 500


 42%|████▏     | 1000/2381 [26:21<38:49,  1.69s/it]

Saved progress at row 1000


...

Saved progress at row 1500


 84%|████████▍ | 2000/2381 [52:27<10:20,  1.63s/it]

Saved progress at row 2000


100%|██████████| 2381/2381 [1:02:24<00:00,  1.57s/it]


Matched 2239 / 2381 songs


In [ ]:
# merge on billboard_index
df_merged = df_filtered.merge(
    df_mb, left_index=True, right_on="billboard_index", how="left"
)

print(f"Merged dataset: {len(df_merged)} songs")
print(df_merged.head(2))

Merged dataset: 2381 songs
          date  rank                                      song  \
0.0 1990-08-11     9                        She Ain't Worth It   
1.0 1984-08-04    97  Against All Odds (Take A Look At Me Now)   

                                   artist  last_week  peak_rank  \
0.0  Glenn Medeiros Featuring Bobby Brown        5.0          1   
1.0                          Phil Collins       84.0          1   

     weeks_on_board  year  decade                                 mb_id  \
0.0              13  1990    1990  1b6a1b61-4754-4a4f-a0fc-89b9dbfb3b85   
1.0              24  1984    1980  161eac8b-ab29-4b8e-b7f1-e2a0631ae660   

                                     mb_title       mb_artist mb_country  \
0.0                        She Ain't Worth It  Glenn Medeiros         AU   
1.0  Against All Odds (Take a Look at Me Now)    Phil Collins         XE   

    mb_date  mb_score  billboard_index  
0.0             100.0                0  
1.0    1990     100.0              

In [ ]:
def create_song_chunk(row):
    parts = [
        f"'{row['song']}' by {row['artist']} ({int(row['year'])})",
        f"Billboard Hot 100 peak: #{int(row['peak_rank'])}",
    ]
    if pd.notna(row.get('mb_country')) and row['mb_country']:
        parts.append(f"Country of origin: {row['mb_country']}")
    if pd.notna(row.get('mb_date')) and row['mb_date']:
        parts.append(f"Release date: {row['mb_date']}")
    if pd.notna(row.get('mb_score')) and row['mb_score']:
        parts.append(f"Match confidence: {int(row['mb_score'])}/100")

    return ". ".join(parts)

df_merged["text_chunk"] = df_merged.apply(create_song_chunk, axis=1)

# check if it looks clean now
for chunk in df_merged["text_chunk"].head(3):
    print(chunk)
    print()

'She Ain't Worth It' by Glenn Medeiros Featuring Bobby Brown (1990). Billboard Hot 100 peak: #1. Country of origin: AU. Match confidence: 100/100

'Against All Odds (Take A Look At Me Now)' by Phil Collins (1984). Billboard Hot 100 peak: #1. Country of origin: XE. Release date: 1990. Match confidence: 100/100

'Me And Mrs. Jones' by Billy Paul (1973). Billboard Hot 100 peak: #1. Country of origin: DE. Release date: 1994. Match confidence: 100/100



In [ ]:
df_merged.to_csv(DATA_PATH + "knowledge_base.csv", index=False)
print(f"Knowledge base saved: {len(df_merged)} songs")
print(f"Sample chunk:\n{df_merged['text_chunk'].iloc[0]}")

Knowledge base saved: 2381 songs
Sample chunk:
'She Ain't Worth It' by Glenn Medeiros Featuring Bobby Brown (1990). Billboard Hot 100 peak: #1. Country of origin: AU. Match confidence: 100/100
